# A2: Presentation robustness

**Question (RQ1B).** Do the decision models give the same answer when the same request is presented differently, beyond the change between two unchanged replays? And does averaging over presentations give better probabilities than the standard presentation?

Each dataset uses up to 500 sampled requests per factor. Unchanged replays are sampled from the union of the wording and field-order samples. Replay-adjusted effects use the intersection with each factor and report its size. One factor changes at a time: three reviewed rewordings of the question sentence, reversed text-field order (joint requests with at least two non-empty fields) and reversed option order (multiclass, realigned by class identity); three unchanged executions estimate replay noise. All tables come from **one compact export**, `output_JEVPFN/A/exp02_robustness/results/robustness/robustness.<cohort>-<roster digest>.zip` (schema `jevpfn.rq1b.export.v3`), built by `scripts/slurm/rq1_robustness_export.slurm`. The historical 0A sensitivity probes travel in the same ZIP but are shown in the general qualification notebook (`general/3_model_and_feature_qualification`).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.evaluation import group_a, learner_scores, model_properties, study_export, study_reports
from src.utils.notebook_display import display_json, display_table
from src.utils.notebook_report import begin_report, finish_report, record, report_section
from src.utils.provenance import provenance
from src.visualize import critical_difference, figures, style
from src.visualize.group_b import render
from src.visualize.stability import (
    averaging_skill_forest,
    excess_change_figure,
    wording_averaging_figure,
)

style.apply()
NOTEBOOK = "A/exp02_robustness"
begin_report()
from src.evaluation.benchmark_export import cohort_record
cohort = cohort_record()
ROSTER = cohort["decision_roster"]
record(cohort, "Decision-model cohort and members")
save = figures.FigureSaver(NOTEBOOK)
report_section("1. What is complete")
display_json("Notebook execution identity", provenance())
frames, receipt = study_reports.robustness_export_tables()
frames = group_a.cohort_frames(frames, ROSTER)
print(study_export.describe(receipt))
record({k: receipt.get(k) for k in ("status", "path", "created_utc", "origin", "sha256", "bytes", "schema", "receipt_sha256")}, "Export identity")
properties = model_properties.profile_table().loc[lambda t: t.provider.isin(ROSTER)]
order = group_a.model_order(properties.provider, properties)
if frames:
    display_table("Completed campaigns in the export", frames["sources"], preview_rows=8)
    display_table("Coverage, including field orders normalized by the model", frames["coverage"], preview_rows=8)
    record(frames["coverage"], "Coverage per campaign")
if frames and not frames["averaging_quality"].empty:
    denominators = frames["averaging_quality"][["dataset", "provider", "factor", "requests", "rows"]].drop_duplicates()
    display_table("Matched joint requests and linked rows per presentation factor", denominators, preview_rows=12)

Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\A\exp02_robustness\results\robustness\robustness.open_weight-5baeccd37d70d034.zip is absent. No measurements are shown.


## 2. Summary: change beyond replay noise

Per model, request mode and factor: the mean total variation distance between the fresh original and the changed presentation, minus the mean distance between the original and its two unchanged replays on the same requests (0 = replay noise). The export measures the full probability vector with total variation for every task (regression: the nine ordered levels); it does not export an earth mover's distance per factor. Datasets count equally; intervals resample datasets.

In [2]:
report_section("2. Summary: change beyond replay noise")
if not frames:
    print("Waiting for export: no stability tables are available.")
else:
    excess = group_a.excess_summary(frames["mode_effects"])
    render(save, excess_change_figure(excess, order=order))
    display_table("Excess change over replay per model, mode and factor (equal dataset weight, dataset-bootstrap 95% CI)", excess.round(4), preview_rows=12)
    record(excess, "Excess change over replay, equal dataset weight")
    by_dataset = group_a.excess_by_dataset(frames["mode_effects"])
    display_table("Matched replay requests per variant and presentations across variants", by_dataset[
        ["dataset", "provider", "mode", "factor", "min_matched_requests_per_variant",
         "max_matched_requests_per_variant", "matched_presentations"]], preview_rows=12)
    record(by_dataset, "Excess change per dataset")
    record(study_reports.equal_dataset_mode_effects(frames["mode_effects"]), "Mean change, excess and top-answer flip rate per model, mode and variant")

Waiting for export: no stability tables are available.


## 3. Does averaging over presentations help?

No model is called again; the responses above are reused. **Averaged versus standard presentation** (primary, not guaranteed): stand-alone quality on the sampled joint requests of the mean over the factor's presentations against the production presentation, as proper-score skill 1 − S_averaged / S_standard with each dataset's own primary proper score (log loss for classification, ranked probability score for regression), so all 18 datasets share one scale. The averaged-versus-mean-single contrast is the Jensen gap, never negative for convex proper scores, and is reported in the table only. Below, the wording-distance view shows how much averaging two wordings reduces wording sensitivity.

In [3]:
report_section("3. Averaging over presentations")
framework = learner_scores.run_application("A2_stability")
if not frames:
    print("Waiting for export: no averaging tables are available.")
else:
    skill_by_dataset = group_a.averaging_skill_by_dataset(frames["averaging_quality"])
    skill = group_a.averaging_skill_summary(frames["averaging_quality"])
    render(save, averaging_skill_forest(skill, skill_by_dataset, order=order, tests=group_a.a2_margin_tests(framework)))
    display_table("Proper-score skill of the averaged over the standard presentation (equal dataset weight, dataset-bootstrap 95% CI)", skill.round(4), preview_rows=12)
    quality = study_reports.averaging_quality_summary(frames["averaging_quality"])
    display_table("Every averaging contrast and metric (primary: averaged vs standard; Jensen gap: averaged vs mean single, guaranteed for convex scores)", quality.round(4), preview_rows=12)
    wording = study_reports.wording_averaging_summary(frames["wording_averaging"])
    render(save, wording_averaging_figure(wording))
    record(skill, "Averaging skill per model and factor")
    record(quality, "Averaged-presentation quality, equal dataset weight")
    record(wording, "Wording averaging, equal dataset weight")

Waiting for export: no averaging tables are available.


## 4. Statistics

The averaged and canonical presentations are compared using proper-score skill. Within each presentation factor, Wilcoxon tests across eligible datasets are Holm-adjusted across the selected models. These are study-level results. The average versus mean-single comparison reports a descriptive Jensen gap. The primary cross-study family is reported in general/5_synthesis.

In [4]:
report_section("4. Statistics")
print(f"A2_stability: {framework['status']} (framework.status = {framework['framework_status']})")
display_json("Framework application A2_stability", learner_scores.summary_record(framework))
render(save, critical_difference.cd_figures(framework))

A2_stability: awaiting_export (framework.status = approved)


## Complete report

In [5]:
finish_report(NOTEBOOK, save.summary() + "\nOnly the export ZIP and the model-properties configuration were read; no inference, model training or VSC changes were performed.")

A/exp02_robustness

Decision-model cohort and members
{
  "decision_cohort": "open_weight",
  "decision_roster": [
    "liquid",
    "liquid_lfm3",
    "perplexity",
    "cloudflare",
    "strands",
    "intern",
    "h2o",
    "sii",
    "kev",
    "decision3",
    "nimble",
    "glide",
    "mammoth",
    "drex_max",
    "knowline",
    "sieve"
  ],
  "decision_models": 16,
  "decision_model_versions": [
    {
      "provider": "liquid",
      "model": "LiquidAI/d1-3B",
      "version": "051bcc464b01b9f92942b364d9586b0ef5912432",
      "delivery": "self_hosted"
    },
    {
      "provider": "liquid_lfm3",
      "model": "LiquidAI/d1-2B",
      "version": "cfe5760ff5d5998f91d9ac56bd4b2dbd96996d41",
      "delivery": "self_hosted"
    },
    {
      "provider": "perplexity",
      "model": "perplexity-ai/pplx-decider-v1.1-27b",
      "version": "14c9d90137962d300f3de95633f7b6535e80c5ce",
      "delivery": "self_hosted"
    },
    {
      "provider": "cloudflare",
      "model": "Cloud